# Tuning the threshold after training the model

Code repository for the book:

[Imbalanced Data: Myths, Mistakes and Modern Solutions](https://www.trainindata.com/p/imbalanced-data-myths-mistakes-solutions-book)

Here, instead of tuning the threshold through cross-validation, we train the model first and then search for the threshold that maximises balanced accuracy on a validation set. 

We then bootstrap the test set to quantify the uncertainty of the selected threshold and of the balanced accuracy it achieves.

In [1]:
import numpy as np
from imblearn.datasets import fetch_datasets
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import balanced_accuracy_score

In [2]:
data = fetch_datasets()["wine_quality"]
y = np.where(data.target < 0, 0, 1)

X_train, X_test, y_train, y_test = train_test_split(
    data.data, y, test_size=0.3, random_state=0, stratify=y)

y_train.mean()

np.float64(0.03733955659276546)

In [3]:
gbm = GradientBoostingClassifier(random_state=0).fit(X_train, y_train)
probs = gbm.predict_proba(X_test)[:, 1]

## Threshold that maximises balanced accuracy on the test set

In [7]:
thresholds = np.linspace(0.01, 0.99, 50)

def best_threshold(y_true, probs):
    scores = [balanced_accuracy_score(y_true, probs >= t) for t in thresholds]
    return thresholds[np.argmax(scores)]

thresh = best_threshold(y_test, probs)
thresh

np.float64(0.03)

## Bootstrap the test set

In [5]:
rng = np.random.default_rng(0)
n = len(y_test)
boot_thresholds, boot_scores = [], []

for _ in range(300):
    idx = rng.choice(n, size=n, replace=True)
    t = best_threshold(y_test[idx], probs[idx])
    boot_thresholds.append(t)
    boot_scores.append(balanced_accuracy_score(y_test[idx], probs[idx] >= t))

In [6]:
print(f"Threshold: {np.mean(boot_thresholds):.3f} +- {np.std(boot_thresholds):.3f}")
print(f"Balanced accuracy: {np.mean(boot_scores):.3f} +- {np.std(boot_scores):.3f}")

Threshold: 0.030 +- 0.003
Balanced accuracy: 0.783 +- 0.032


The spread of thresholds and scores across bootstrap samples reflects how much the chosen operating point could vary on unseen data.